# Train a Tiny Story-Writing Language Model, Step by Step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/tinystories-ARM-model/blob/main/train_step_by_step.ipynb)

This notebook builds a small GPT-style language model (~30M parameters) from scratch and trains it on
[TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories), a dataset of simple short stories.

Run the cells **top to bottom**. Each step explains what it does and shows you something you can look at.

> This project is built for **Apple Silicon Macs**, where it trains on the Mac's GPU and comes with a
> step-by-step web app (`python3 app/server.py`). This notebook is the option for anyone without one:
> it runs on Colab's free Nvidia GPU.

| Step | What happens |
|---|---|
| 0 | Setup: get the code, pick a device |
| 1 | Download the dataset (only once) |
| 2 | Tokenize: turn text into numbers |
| 3 | Make training batches |
| 4 | Build the model |
| 5 | Choose training settings |
| 6 | Train |
| 7 | Plot the loss curves |
| 8 | Generate stories |
| 9 | Inference metrics on the GPU: the KV cache |

**On Colab:** use a GPU for much faster training: *Runtime → Change runtime type → T4 GPU*.

## Step 0 · Setup

On Colab this clones the repository and installs the dependencies. Locally, open this notebook from the
repository folder (after `pip install -r requirements.txt`) and it uses the code that's already there.

In [ ]:
# @title Get the code
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/tinystories-ARM-model"):
        !git clone -q {REPO_URL} /content/tinystories-ARM-model
    %cd /content/tinystories-ARM-model
    !pip install -q -r requirements.txt

assert os.path.exists("full-model.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
print("Project folder:", PROJECT_DIR)

In [ ]:
# @title Import the project modules and pick a device
import torch

from data import BatchLoader, dataset_paths, get_dataset
from inference import generate_text, load_model, save_model_config
from model import GPT, GPTConfig, estimate_loss
from plots import plot_losses
from pretrain import pretrain
from tokenization import bin_files_exist, get_tokenizer, tokenize_and_save
from train_config import TrainConfig, setup_device

dev = setup_device()
print(f"Device: {dev.device}  |  precision: {dev.dtype}")
if dev.device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  |  memory: {props.total_memory / 1e9:.1f} GB  |  compute capability {props.major}.{props.minor}")
    !nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
elif dev.device == "cpu":
    print("No GPU found: training will be slow. On Colab, switch to a GPU runtime.")

## Step 1 · Download the dataset

The default is TinyStories: about 2.1 million short stories (~1.8 GB on disk). You can switch to any text
dataset: a Hugging Face name (e.g. `Salesforce/wikitext` with subset `wikitext-2-raw-v1`), a Hugging Face
link, or a link to a `.txt`, `.jsonl`, `.csv` or `.parquet` file, such as
`https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt`.
Only text datasets are accepted. If there's no validation split, a small slice of train is held out.
Run `python3 full-model.py --list-datasets` for more datasets that work out of the box.

The first run downloads the dataset and saves a copy to `data/<dataset>/`; later runs load that copy instead
of downloading again. Each dataset keeps its own tokens and trained model in that folder.

In [ ]:
# @title Load the dataset
dataset_name = "roneneldan/TinyStories"  # @param {type:"string"}
dataset_config = ""  # @param {type:"string"}
text_column = "text"  # @param {type:"string"}

paths = dataset_paths(dataset_name, dataset_config or None, os.path.join(PROJECT_DIR, "data"))
ds = get_dataset(paths.hf_dir, dataset_name, config_name=dataset_config or None, text_column=text_column)
print("Stored in:", os.path.relpath(paths.root, PROJECT_DIR))
print(ds)

In [ ]:
# @title Peek at a story
story_index = 0  # @param {type:"integer"}
print(ds["train"][story_index]["text"])

## Step 2 · Tokenize

Models work with numbers, not letters. The GPT-2 tokenizer splits text into **tokens** (whole words or
pieces of words) and gives each one an ID from a vocabulary of 50,257.

Every story is tokenized once and all the IDs are written into two files, `train.bin` and `validation.bin`.
If they already exist this step is skipped. *Tokenizing the full dataset takes a while the first time,
especially on Colab's small CPUs.*

In [ ]:
# @title Try the tokenizer
enc = get_tokenizer()
sample_text = "Once upon a time, there was a little cat named Tom."  # @param {type:"string"}
ids = enc.encode_ordinary(sample_text)
print("Token IDs:", ids)
print("Tokens:   ", [enc.decode([i]) for i in ids])
print(f"{len(sample_text)} characters -> {len(ids)} tokens")

In [ ]:
# @title Tokenize the whole dataset (skipped if already done)
import numpy as np

if bin_files_exist(paths.root):
    print("train.bin and validation.bin already exist, skipping.")
else:
    tokenize_and_save(ds, paths.root, num_proc=os.cpu_count())

for f in (paths.train_bin, paths.val_bin):
    n = len(np.memmap(f, dtype=np.uint16, mode="r"))
    print(f"{os.path.basename(f)}: {n:,} tokens")

## Step 3 · Make training batches

Training doesn't read stories one by one. Each **batch** is a set of random windows of `block_size`
consecutive tokens cut from `train.bin`. The **target** is the same window shifted one token to the right:
at every position, the model learns to predict the next token.

`block_size` is the model's *context window*: how many tokens it can look at once.

In [ ]:
# @title Batch settings
block_size = 128  # @param {type:"integer"}
batch_size = 32  # @param {type:"integer"}
seed = 42  # @param {type:"integer"}

torch.manual_seed(seed)
loader = BatchLoader(paths.root, block_size, batch_size, dev.device)
X, Y = loader.get_batch("train")
print("Inputs X:", tuple(X.shape), " Targets Y:", tuple(Y.shape), " (batch_size, block_size)")
print()
print("Input :", repr(enc.decode(X[0, :24].tolist())))
print("Target:", repr(enc.decode(Y[0, :24].tolist())))

## Step 4 · Build the model

A decoder-only transformer, like a miniature GPT-2:

- **Embeddings** turn each token ID and its position into a vector of size `n_embd`.
- **`n_layer` transformer blocks**, each with causal self-attention (`n_head` heads; every token looks only at
  earlier tokens) and a small feed-forward network.
- A final layer scores every word in the vocabulary as the possible next token.

Bigger numbers mean a more capable model, but slower training and more memory.

In [ ]:
# @title Model settings
# Presets: tiny 16M · small 30M (default) · medium 51M · large 124M (GPT-2 small size; fits easily on a T4)
model_size = "small"  # @param ["tiny", "small", "medium", "large", "custom"]
n_layer = 6  # @param {type:"integer"}
n_head = 6  # @param {type:"integer"}
n_embd = 384  # @param {type:"integer"}
dropout = 0.1  # @param {type:"number"}

PRESETS = {"tiny": (4, 4, 256), "small": (6, 6, 384), "medium": (8, 8, 512), "large": (12, 12, 768)}
if model_size != "custom":
    n_layer, n_head, n_embd = PRESETS[model_size]

config = GPTConfig(
    vocab_size=enc.n_vocab,
    block_size=block_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    dropout=dropout,
    bias=True,
)
torch.manual_seed(seed)
model = GPT(config).to(dev.device)
n_params = sum(p.numel() for p in model.parameters())
print(config)
print(f"Parameters: {n_params / 1e6:.1f}M")

In [ ]:
# @title Before training: loss and a sample
import math

with torch.no_grad():
    _, loss = model(X, Y)
print(f"Loss on one batch: {loss.item():.2f}")
print(f"Random guessing would score ln({enc.n_vocab}) = {math.log(enc.n_vocab):.2f}")
print()
model.eval()
print(generate_text(model, enc, "Once upon a time", dev.device, max_new_tokens=30))
model.train();

Gibberish, as expected: the weights are still random. Training fixes that.

## Step 5 · Training settings

- **quick**: a short run to see the whole pipeline work. The loss drops noticeably, but the stories will
  still be rough.
- **full**: the settings `full-model.py` uses (20,000 steps). Expect this to take hours.

Each optimizer update averages `gradient_accumulation_steps` batches, so the model sees
`batch_size × gradient_accumulation_steps` windows per update. The learning rate warms up for
`warmup_steps`, then decays along a cosine curve down to `min_lr`.

In [ ]:
# @title Choose a preset
mode = "quick"  # @param ["quick", "full"]
learning_rate = 1e-4  # @param {type:"number"}

best_model_path = paths.checkpoint

if mode == "quick":
    train_cfg = TrainConfig(
        max_iters=2000, warmup_steps=200, eval_interval=250, eval_iters=50,
        gradient_accumulation_steps=8,
        learning_rate=learning_rate, batch_size=batch_size, seed=seed, best_model_path=best_model_path,
    )
else:
    train_cfg = TrainConfig(
        learning_rate=learning_rate, batch_size=batch_size, seed=seed, best_model_path=best_model_path,
    )
train_cfg

## Step 6 · Train

Every `eval_interval` steps, the loss is measured on training and validation data. Whenever validation loss
reaches a new low, the weights are saved to `best_model_params.pt`.

Watch both numbers go down. If validation loss starts rising while training loss keeps falling, the model is
memorizing instead of learning (*overfitting*).

> Running this cell again continues training the same model. To start over, re-run the
> *Model settings* cell in Step 4 first.

In [ ]:
# @title Train the model
import time
from dataclasses import asdict

loader = BatchLoader(paths.root, config.block_size, train_cfg.batch_size, dev.device)
if dev.device == "cuda":
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_losses, val_losses = pretrain(model, loader, train_cfg, dev)
seconds = time.time() - t0
best_val = min(val_losses) if val_losses else None
tokens = train_cfg.max_iters * train_cfg.batch_size * config.block_size
updates = -(-train_cfg.max_iters // train_cfg.gradient_accumulation_steps)
print(f"Trained {train_cfg.max_iters:,} steps in {seconds / 60:.1f} min: {tokens / seconds:,.0f} tokens/s (including evaluations)")
print(f"{updates:,} optimizer updates × {n_params:,} parameters = {updates * n_params:,} parameter updates")
if dev.device == "cuda":
    print(f"Peak GPU memory while training: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB "
          f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
save_model_config(paths.model_config, config, {
    "dataset": dataset_name, "dataset_config": dataset_config or None, "steps": train_cfg.max_iters,
    "best_val": best_val, "trained_at": time.strftime("%Y-%m-%d %H:%M"), "train_config": asdict(train_cfg)})
print(f"Best validation loss: {best_val:.4f}" if best_val is not None else "No evaluation ran (max_iters < eval_interval).")

## Step 7 · Plot the loss curves

In [ ]:
# @title Training vs validation loss
plot_losses(train_losses, val_losses, train_cfg.eval_interval)

## Step 8 · Generate stories

This loads the checkpoint with the best validation loss and continues your prompt.

- **temperature**: lower is safer and more repetitive, higher is more surprising (and more random).
- **top_k**: only pick among the `k` most likely next tokens. `0` means no limit.

Change the prompt and run the cell as many times as you like.

In [ ]:
# @title Write a story
prompt = "Once upon a time there was a little girl who"  # @param {type:"string"}
max_new_tokens = 200  # @param {type:"integer"}
temperature = 0.8  # @param {type:"slider", min:0.1, max:2.0, step:0.1}
top_k = 50  # @param {type:"integer"}

trained = load_model(config, train_cfg.best_model_path, dev.device)
print(generate_text(trained, enc, prompt, dev.device, max_new_tokens=max_new_tokens,
                    temperature=temperature, top_k=top_k or None))

## Step 9 · Inference on the GPU: the KV cache

Generating text is one forward pass per new token. Without a cache, each pass re-reads the whole context.
The **KV cache** keeps every layer's keys and values for the tokens already seen, so each new pass only
processes the new token. This cell generates the same text both ways (same random seed) and compares speed
and memory on this GPU, for a few lengths.

The cache costs `2 (key + value) × n_layer × n_embd × 4 bytes` per token. This model's positions are learned
per slot, so once the context window (`block_size`) is full the cache has to be rebuilt every step and the
advantage shrinks: watch the last row.

With a small model and a 128-token window, each step is dominated by the fixed cost of launching GPU work,
so expect a modest gain (on an Apple M5: about 1.1× for the Small model, 1.3× for Large). Try `model_size = "large"` in Step 4: the bigger
the model and the longer the context, the more the cache saves.

In [ ]:
# @title Benchmark generation with and without the KV cache
from inference import benchmark_generation, device_memory, kv_bytes_per_token

prompt_ids = enc.encode_ordinary("Once upon a time there was a little girl who")
per_token = kv_bytes_per_token(trained.config)
print(f"KV cache: {per_token / 1024:.1f} KB per token, {per_token * trained.config.block_size / 1e6:.2f} MB for a full "
      f"{trained.config.block_size}-token window  |  weights: {n_params * 4 / 1e6:.0f} MB")
print()
print(f"{'tokens':>7} | {'no cache tok/s':>14} | {'KV cache tok/s':>14} | {'speedup':>7} | {'cache':>8} | {'rebuilds':>8} | same text")
for n in (25, 60, 110, 250):
    r = benchmark_generation(trained, prompt_ids, dev.device, n_tokens=n)
    a, b = r["no_cache"], r["kv_cache"]
    print(f"{n:>7} | {a['tokens_per_sec']:>14} | {b['tokens_per_sec']:>14} | {r['speedup']:>6}× | "
          f"{b['kv_cache_bytes'] / 1e6:>6.2f}MB | {b['recomputes']:>8} | {r['same_output']}")
mem = device_memory(dev.device)
if mem.get("allocated") is not None:
    print(f"\nGPU memory in use: {mem['allocated'] / 1e6:.0f} MB" + (f", peak {mem['peak'] / 1e6:.0f} MB" if mem.get("peak") else ""))

## Next steps

- Try the **full** preset for much better stories, and a bigger model (`model_size = "large"`, 124M parameters) in Step 4.
- Compare the loss curves and the Step 9 speeds for different model sizes.
- Try another dataset in Step 1 and compare.
- On Colab, download your trained weights: *Files panel → `data/<dataset>/best_model_params.pt` → Download*.
- To run everything without the notebook: `python3 full-model.py`.